# fine-tune + retrieval in Colab
**Before running:** Runtime → Change runtime type → **T4 GPU**.

How it works:
1. Fine-tune a small pretrained LLM (Qwen2.5-0.5B-Instruct) on your `book.txt` with LoRA, so it absorbs the book's content and style.
2. At question time, retrieve the most relevant passages from the book and feed them to the model, which keeps answers accurate and grounded.

here we are fine tuining a pretrained qwen model to know about mahabharata that is inside "Mahabharata_Complete.txt" file

In [ ]:
!pip install -q -U transformers peft datasets accelerate sentence-transformers faiss-cpu gradio
'''
-q stands for quiet. It tells pip to print much less text while installing.
Normally, pip install shows a lot of output: “Collecting transformers...”, download progress
bars, “Installing collected packages...”, and so on. With -q, most of that is hidden, so your
notebook cell stays short and clean. You’ll only see warnings or errors if something goes wrong.
'''
#-U means upgrade, so any of these packages already installed get updated to the latest version.
# installs transformers peft datasets accelerate sentence-transformers faiss-cpu gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 112.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 81.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 18.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


## 1. Upload book.txt

In [ ]:
#from google.colab import files
#files.upload()   # choose Mahabharata_Complete.txt
text = open("Mahabharata_Complete.txt", encoding="utf-8", errors="ignore").read()
print(f"{len(text.split()):,} words")

2,548,247 words


## 2. Load base model

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

#model name
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"   # for better answers try "Qwen/Qwen2.5-1.5B-Instruct" (slower)
tok = AutoTokenizer.from_pretrained(MODEL)
'''
Models can’t read text directly, only numbers.
The tokenizer converts text into token IDs (numbers representing words or word pieces) and converts the model’s output numbers back into text.
from_pretrained downloads the correct tokenizer for this specific model. Auto means it figures out the right tokenizer class automatically.
'''
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32).to("cuda")  #loads a pretrained language model and puts it on the GPU

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

## 3. Fine-tune on the book (LoRA)

In [ ]:
!pip install -U "torchao>=0.16.0"  #upgrades PyTorch’s quantization library (torchao) to version 0.16.0 or newer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 27.9 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import Dataset

BLOCK = 512
ids = []
for i in range(0, len(text), 100_000):                      # tokenize in pieces
    ids += tok(text[i:i+100_000], add_special_tokens=False)["input_ids"]
n = len(ids) // BLOCK * BLOCK
blocks = [ids[i:i+BLOCK] for i in range(0, n, BLOCK)]
ds = Dataset.from_dict({"input_ids": blocks, "attention_mask": [[1]*BLOCK]*len(blocks)})
print(len(blocks), "training blocks")

cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                 target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
model = get_peft_model(model, cfg)
model.print_trainable_parameters()

args = TrainingArguments(output_dir="out", per_device_train_batch_size=4, gradient_accumulation_steps=4,
                         num_train_epochs=2, learning_rate=2e-4, lr_scheduler_type="cosine",
                         warmup_steps=0.03, fp16=True, logging_steps=20,
                         save_strategy="no", report_to="none")
Trainer(model=model, args=args, train_dataset=ds,
        data_collator=DataCollatorForLanguageModeling(tok, mlm=False)).train()
model.save_pretrained("book-lora")

7431 training blocks


/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:148: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:331: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


Step,Training Loss
20,3.112308
40,2.925924
60,2.810592
80,2.757446
100,2.729518
120,2.705237
140,2.686052
160,2.641992
180,2.632151
200,2.645704


## 4. Build the retrieval index

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss, numpy as np

words = text.split()
CH, OVERLAP = 200, 40
chunks = [" ".join(words[i:i+CH]) for i in range(0, len(words), CH-OVERLAP)]
emb = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda")
E = emb.encode(chunks, batch_size=128, normalize_embeddings=True, show_progress_bar=True).astype("float32")
index = faiss.IndexFlatIP(E.shape[1]); index.add(E)
print(len(chunks), "chunks indexed")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/125 [00:00<?, ?it/s]

15927 chunks indexed


## 5. Ask questions

In [ ]:
model.eval()

def ask(q, k=4, max_new_tokens=300):
    qv = emb.encode([q], normalize_embeddings=True).astype("float32")
    _, I = index.search(qv, k)
    ctx = "\n\n".join(chunks[i] for i in I[0])
    msgs = [
        {"role": "system", "content": "Answer the question using the book excerpts. If the answer is not in them, say you don't know."},
        {"role": "user", "content": f"Excerpts:\n{ctx}\n\nQuestion: {q}"},
    ]
    inp = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True).to("cuda")
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.1)
    return tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)


This book is about the history of the Vrishnis, the Pandavas, and the Kurus. It describes how the three orders (Vrishni, Pandava, and Kuru) have been created and how they have come to be known by different names. The text mentions the birth of the Pandavas, the creation of the Vrishni race, the rise of the Pandavas, and the end of the Kurus. It also recounts the story of the battle between the Kurus and the Pandavas. The narrative covers the period of the great battle between the Kurus and the Pandavas, including the slaughter of the Srinjayas and the slaughter of the Pandavas themselves.


## 6. Chat UI

In [ ]:
import gradio as gr
gr.ChatInterface(lambda message, history: ask(message), title="Ask the Book").launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b4454064fbd0535045.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# 7. Download the model

In [ ]:
#mount drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#Merge and save (PEFT / transformers)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE = "Qwen/Qwen2.5-0.5B-Instruct"          # the same base model you trained the LoRA on
ADAPTER = "book-lora"                  # your saved adapter folder
OUT = "/content/drive/MyDrive/book-lora-merged"

# Load the base in fp16/bf16 (NOT 4-bit/8-bit, or the merge will be lossy)
base = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16, device_map="auto")
tok = AutoTokenizer.from_pretrained(BASE)

model = PeftModel.from_pretrained(base, ADAPTER)
merged = model.merge_and_unload()      # folds the LoRA weights into the base weights

merged.save_pretrained(OUT, safe_serialization=True)
tok.save_pretrained(OUT)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/peft/peft_model.py:642: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight', 'base_model.model.model.layers.0.mlp.gate_proj.lora_A.default.weight', 'base_model.model.model.layers.0.mlp.gate_proj.lora_B.default.weight', 'base_model.model.model.layers.0.mlp.up_proj.lora_A.default.weight', 'base_model.model.model.layers.0.mlp.up_proj.lora_B.default.we

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/book-lora-merged/tokenizer_config.json',
 '/content/drive/MyDrive/book-lora-merged/chat_template.jinja',
 '/content/drive/MyDrive/book-lora-merged/tokenizer.json')

# 8. Reload the model

In [ ]:
#loading model
from transformers import AutoModelForCausalLM, AutoTokenizer
path = "/content/drive/MyDrive/book-lora-merged"
model = AutoModelForCausalLM.from_pretrained(path, torch_dtype="auto", device_map="auto")
tok = AutoTokenizer.from_pretrained(path)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]